<a href="https://colab.research.google.com/github/CarlosNoriegaPolo/fMetRecogninPrediction/blob/colab_development/fMet_Recognin_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **fMet Recognin Prediction Project**

### **Coding Rules for This Project**

*   **Write Clear Code**: Use meaningful variable and function names.

*   **Comment Your Code**: Briefly explain your code and reasoning.

*   **Keep It Organized**: Use different sections to develop and test individual portions of the code.

*   **Incorporate Checks**: Frequently check the output of code blocks by printing statements and execution times

# Part 0. Load libraries

Load all the neccesary libraries here.

In [1]:
import requests # for API requests
import re # regular expressions
import time # to control running times

# Part 1. Retrieve Protein Models - Data Mining

## Get all reviewed human proteins from UniProt API

In [3]:
# Set parameters
batch_size = 500 # it will retrieve entries in groups of 500 (can change)
base_url = "https://rest.uniprot.org/uniprotkb/search"
re_next_link = re.compile(r'<(.+)>; rel="next"')

# Set up session with retry strategy
session = requests.Session()
retry_strategy = requests.adapters.Retry(
    total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504] # these are the codes for failed attempts
)
session.mount("https://", requests.adapters.HTTPAdapter(max_retries=retry_strategy))

# Construct query parameters (explicitly requesting only the necessary fields)
params = {
    'query': '(reviewed:true) AND (organism_id:9606)',  # 9606 is the taxonomy ID for humans
    'format': 'tsv',
    'fields': 'accession,protein_name',  # Request only UniProt ID and protein name
    'size': batch_size
}

# Construct initial URL
url = f"{base_url}?{'&'.join(f'{k}={v}' for k, v in params.items())}"

# Initialize dictionary to store protein data
protein_IDs = {}
progress = 0

# Start timer for total time
start_time = time.time()

# Fetch and process batches
while url:
    try:
        # Send request and get response
        response = session.get(url)
        response.raise_for_status()

        # Parse the response
        lines = response.text.splitlines()

        # Process the data (skip header)
        for line in lines[1:]:  # Skip the header
            fields = line.split("\t")
            if len(fields) >= 2:  # Ensure there are at least two columns
                uniprot_id, protein_name = fields[:2]  # Take only the first two values
                protein_IDs[protein_name] = uniprot_id

        # Update progress
        progress += len(lines[1:])
        total = response.headers.get("x-total-results", "unknown")
        print(f"Progress: {progress:,} / {int(total):,} entries",
              f"({progress/int(total)*100:.1f}%)")

        # Extract the next page URL from headers (pagination)
        next_link = response.headers.get("Link")
        if next_link:
            match = re.search(r'<(.+)>; rel="next"', next_link)
            url = match.group(1) if match else None
        else:
            url = None

    except requests.exceptions.RequestException as e:
        print(f"Error fetching batch: {e}")
        break

# Calculate and print total time
duration = time.time() - start_time
print(f"\nDownload completed in {duration:.1f} seconds")
print(f"Total proteins: {len(protein_IDs)}")

Progress: 500 / 20,417 entries (2.4%)
Progress: 1,000 / 20,417 entries (4.9%)
Progress: 1,500 / 20,417 entries (7.3%)
Progress: 2,000 / 20,417 entries (9.8%)
Progress: 2,500 / 20,417 entries (12.2%)
Progress: 3,000 / 20,417 entries (14.7%)
Progress: 3,500 / 20,417 entries (17.1%)
Progress: 4,000 / 20,417 entries (19.6%)
Progress: 4,500 / 20,417 entries (22.0%)
Progress: 5,000 / 20,417 entries (24.5%)
Progress: 5,500 / 20,417 entries (26.9%)
Progress: 6,000 / 20,417 entries (29.4%)
Progress: 6,500 / 20,417 entries (31.8%)
Progress: 7,000 / 20,417 entries (34.3%)
Progress: 7,500 / 20,417 entries (36.7%)
Progress: 8,000 / 20,417 entries (39.2%)
Progress: 8,500 / 20,417 entries (41.6%)
Progress: 9,000 / 20,417 entries (44.1%)
Progress: 9,500 / 20,417 entries (46.5%)
Progress: 10,000 / 20,417 entries (49.0%)
Progress: 10,500 / 20,417 entries (51.4%)
Progress: 11,000 / 20,417 entries (53.9%)
Progress: 11,500 / 20,417 entries (56.3%)
Progress: 12,000 / 20,417 entries (58.8%)
Progress: 12,500 

In [9]:
# Check the first 10 items of the protein_IDs dictionary
for i, (protein_name, uniprot_id) in enumerate(protein_IDs.items()):
    if i < 10:
        print(f"Protein ID: {uniprot_id} Protein Name: {protein_name}")

Protein ID: A0A0C5B5G6 Protein Name: Mitochondrial-derived peptide MOTS-c (Mitochondrial open reading frame of the 12S rRNA-c)
Protein ID: A0A1B0GTW7 Protein Name: Ciliated left-right organizer metallopeptidase (EC 3.4.24.-) (Leishmanolysin-like peptidase 2)
Protein ID: A0JNW5 Protein Name: Bridge-like lipid transfer protein family member 3B (Syntaxin-6 Habc-interacting protein of 164 kDa) (UHRF1-binding protein 1-like)
Protein ID: A0JP26 Protein Name: POTE ankyrin domain family member B3
Protein ID: A0PK11 Protein Name: Clarin-2
Protein ID: A1A4S6 Protein Name: Rho GTPase-activating protein 10 (GTPase regulator associated with focal adhesion kinase 2) (GRAF2) (Graf-related protein 2) (Rho-type GTPase-activating protein 10)
Protein ID: A1A519 Protein Name: Protein FAM170A (Zinc finger domain-containing protein) (Zinc finger protein ZNFD)
Protein ID: A1L190 Protein Name: Synaptonemal complex central element protein 3 (Testis highly expressed gene 2 protein) (THEG-2)
Protein ID: A1L3X0 P

## Get data for **features** and explore it

Start by using the UniProt IDs stored in the protein_IDs dictionary. Using the UniProt API, extract all information regarding the features for each protein and append this information into a dataframe. Then analyse this further to completely understand our data i.e. what are all of the possible labels, what are the average positions/lengths of each feature, etc.

## Get data for **models** and explore it

Start by using the UniProt IDs stored in the protein_IDs dictionary. Using the UniProt API, extract all information regarding the protein models available for each protein and append this information into a separate dataframe. Then analyse this further to completely understand our data i.e. how many proteins do not have any models, only Alphafold, etc.